# Импрорты и нужные классы

In [ ]:
import sys
import os
import torch
import copy
import numpy as np
from dotenv import load_dotenv

sys.path.append('..')
env_file = os.path.join('..', '.env')
load_dotenv(env_file)
rowGlam_type = os.environ['ROW_GLAM_TYPE'] # custom or base
name_dataset = os.environ['NAME_DATASET']
name_test_dataset = os.environ['NAME_TEST_DATASET']
dataset_path = os.environ['DATASET_PATH']
test_path = os.environ['TEST_PATH']
test_coco_path = os.environ['TEST_COCO_PATH']
coco_path = os.environ['COCO_PATH']
cache_path = os.environ['CASH_PATH']
cache_pdf = os.environ['CASH_PDF_PATH']

KeyError: 'NAME_DATASET'

In [ ]:
from rows2regionsGLAM.utils.pdf_manager import PDFManager
from rows2regionsGLAM.utils.loger import Loger
from rows2regionsGLAM.utils.row_manager import RowManager
from rows2regionsGLAM.utils.ploter import Ploter
from rows2regionsGLAM.utils.trainer import Trainer
from rows2regionsGLAM.utils.tester import Tester
from rows2regionsGLAM.utils.cacher import Cacher
from rows2regionsGLAM.utils.coco_manager import COCOManager
from rows2regionsGLAM.utils.imbalance import calculate_imbalance
from rows2regionsGLAM.utils.tester import collect_maps, print_map_table

from rows2regionsGLAM.models.rowGLAM_base import TorchModelBase, PARAMS_BASE
from rows2regionsGLAM.models.rowGLAM_custom import TorchModel, PARAMS

from rows2regionsGLAM.tokenizers import RowGLAMTokenizer
from rows2regionsGLAM.converters import Rows2Regions
from rows2regionsGLAM.datasetloaders.base_line_dataset import GLAMDataset
from pager.page_model.sub_models.dtype import ImageSegment
from pager.page_model.sub_models import RegionModel, RowsModel

In [ ]:
loger = Loger('log.txt')
pdf_manager = PDFManager(conf={"loger": loger, "pdf_reader": "PDFMiner"})
row_manager = RowManager(conf={"loger": loger, "add_image": True})
coco_manager = COCOManager(conf={"loger": loger, "coco_path": coco_path})
ploter = Ploter(conf={"loger": loger})
tokenizer = RowGLAMTokenizer()
loger(tokenizer.get_name())

pdf2torch_dict = Cacher({
    "loger": loger,
    "pdf_manager": pdf_manager,
    "row_manager": row_manager,
    "tokenizer": tokenizer
})

# Проверка чтения файла

In [2]:
json_true_regions, CLASSES = coco_manager.get_regions_from_json()

NameError: name 'coco_manager' is not defined

In [1]:
CLASSES[3] = "text"
CLASSES

NameError: name 'CLASSES' is not defined

In [ ]:
file_names = list(json_true_regions.keys())
file_names.sort()

In [ ]:
num_file = 1
file_name = file_names[num_file]
dataset_file = os.path.join(dataset_path, file_name)
pdf_json, pdf_img = pdf_manager.get_json_and_img_from_pdf(dataset_file, num_page=0)
row_json = row_manager.get_row_json_from_pdf_json(pdf_json)
torch_dict = tokenizer(row_json, pdf_img)
ploter.set_dpi(200)
ploter.plot_img(pdf_img)
ploter.plot_tokens(tokenizer, torch_dict)

# Сохранение датасета

In [ ]:
dataset = GLAMDataset({
    "loger": loger,
    "pdf_dir": dataset_path,
    "coco_file": coco_path,
    "count_class": len(CLASSES),
    "name_dataset": name_dataset,
    "default_index": 0,
    "cache_dir": cache_pdf,
    "pdf2torch_dict": pdf2torch_dict
})

# Создание Cache
N = len(dataset)
for i, d in enumerate(dataset):
    print(f"{(i+1)/N*100:4.2f} %", end='\r')

# Отрисовска файла из датасета

In [ ]:
torch_dict2 = dataset[100]
path_pdf = os.path.join(dataset_path, torch_dict2['file_name'] + ".pdf")
_, img = pdf_manager.get_json_and_img_from_pdf(path_pdf)
ploter.set_dpi(200)
ploter.plot_img(img)
ploter.plot_tokens(tokenizer, torch_dict2, markup=True)

# Регионы --------------------------------------------------------------
# ann = coco_manager.get_regions_from_json()
# regions = ann[torch_dict2['file_name'] + ".pdf"]['regions']

# for reg in regions:
#     seg = ImageSegment(dict_p_size=reg['segment'])
#     seg.plot()

# Параметры модели

In [ ]:
model_name = 'row2region_GLAM_artic'

if rowGlam_type == "base":
    model_params = copy.deepcopy(PARAMS_BASE)
elif rowGlam_type == "custom":
    model_params = copy.deepcopy(PARAMS)
    
model_params["NodeClasses"] = len(CLASSES)

model_params["epochs"] = 10
model_params["batch_size"] = 64

if rowGlam_type == "custom":
    model_params["model_type"] = 2 # 1 or 2 or 3

In [ ]:
# Дисбаланс классов ------------------------------------
publaynet_imbalance, edge_imbalance = calculate_imbalance(dataset)

print(publaynet_imbalance, edge_imbalance)

model_params['loss_params']['publaynet_imbalance'] = publaynet_imbalance
model_params['loss_params']['edge_imbalance'] = edge_imbalance

model_params

# Обучение модели

In [ ]:
model:torch.nn.Module = TorchModel(model_params)
total_params = sum(p.numel() for p in model.parameters())  
print(f"Number of parameters: {total_params}") 

In [ ]:
trainer_pub = Trainer(conf={"loger": loger, "params": model_params, "model_name": model_name})
trainer_pub.start_train(5, dataset)

# Проверка работы модели

In [ ]:
model_params['sigmoidEdge'] = True

if rowGlam_type == "base":
    model = TorchModelBase(model_params)
elif rowGlam_type == "custom":
    model = TorchModel(model_params)

model.load_state_dict(torch.load(model_name, weights_only=True))

rows_model = RowsModel()
region_model = RegionModel()
rows2regions = Rows2Regions({
    'model':model, 
    'tokenizer': RowGLAMTokenizer(),
    'is_merge_extract': True,
    'classes': CLASSES
})

In [ ]:
torch_dict2 = dataset[126]
path_pdf = os.path.join(dataset_path, torch_dict2['file_name'] + ".pdf")
pdf_json, img = pdf_manager.get_json_and_img_from_pdf(path_pdf)
ploter.set_dpi(200)
ploter.plot_img(img)
# ploter.plot_tokens(tokenizer, torch_dict2)

# Регионы --------------------------------------------------------------
row_json = row_manager.get_row_json_from_pdf_json(pdf_json)

rows_model.from_dict({"rows": row_json})
rows2regions.convert(rows_model, region_model, img)

for reg in region_model.regions:
    reg.segment.plot(text=reg.label)

In [ ]:
test_dataset = GLAMDataset(
    {
    "loger": loger,
    "pdf_dir": test_path,
    "coco_file": test_coco_path,
    "count_class": len(CLASSES),
    "name_dataset": name_dataset,
    "default_index": 0,
    "cache_dir": cache_pdf,
    "pdf2torch_dict": pdf2torch_dict
    }
)

In [ ]:
loger("Start Test")
loger.time_log()

N = len(test_dataset)
for i, d in enumerate(test_dataset):
    print(f"{(i+1)/N*100:4.2f} %", end='\r')

In [ ]:
tester = Tester(conf={
    "loger": loger, 
    "pdf_manager": pdf_manager, 
    "row_manager": row_manager, 
    "rows_model": rows_model, 
    "region_model": region_model, 
    "rows2regions": rows2regions})

metrics = tester.calculate_target_and_preds(
    test_dataset, 
    name_dataset, 
    name_test_dataset, 
    dataset_path, 
    test_path
)

In [ ]:
tester.print_result(metrics)

In [ ]:
mAP@IoU[0.50:0.95]   :0.52044123
==================================================
threshold_05--------------------
precision_row       :0.8745
recall_row          :0.7442
f1_row              :0.8041
precision_word      :0.8312
recall_word         :0.7130
f1_word             :0.7676
threshold_95--------------------
precision_row       :0.7275
recall_row          :0.6222
f1_row              :0.6707
precision_word      :0.6884
recall_word         :0.5949
f1_word             :0.6382